# 04 · Soul agents — LLM personas for qualitative feedback

Oransim augments the 1M IPF-calibrated agent population with a pool of 10k LLM-driven personas ('soul agents'). These provide qualitative feedback — decision rationale, intent signals, natural-language reactions to the creative — on top of the quantitative world-model prediction.

**This notebook runs in `LLM_MODE=mock` by default (deterministic stub responses, no API calls)**. Set `LLM_API_KEY` environment variable and change `LLM_MODE=api` to use real LLM reactions.

In [ ]:
import os, sys
sys.path.insert(0, '../backend')
os.environ['LLM_MODE'] = 'mock'  # switch to 'api' + set LLM_API_KEY for real LLM

from oransim.data.population import generate_population
from oransim.agents.soul import SoulAgentPool
from oransim.agents.soul_llm import llm_info, llm_available

print('LLM config:', llm_info())
print('LLM active:', llm_available())

In [ ]:
# Generate a small population + soul pool
pop = generate_population(N=5_000, seed=42)
souls = SoulAgentPool(pop, n=8, seed=7)

print(f'population N = {pop.N}')
print(f'soul pool n = {len(souls.personas)}')
print()

# Inspect a persona card
for persona_id, persona in list(souls.personas.items())[:3]:
    print(f'--- persona {persona_id} ---')
    print(persona.full_card()[:300])
    print()

In [ ]:
# Reactions to a creative (mock mode returns deterministic stubs)
creative_caption = 'Aurora morning serum — restocked after the summer sellout'

for persona_id, persona in list(souls.personas.items())[:4]:
    # In mock mode this uses a deterministic hash-based stub;
    # in api mode it would call the configured LLM endpoint.
    reaction = souls.infer_one(persona, creative_caption, platform='xhs')
    print(f'--- persona {persona_id} ---')
    print(f"click:   {reaction.get('click_prob', '?'):.2f}")
    print(f"comment: {reaction.get('comment_prob', '?'):.2f}")
    print(f"why:     {reaction.get('why', '?')[:120]}")
    print()

## What else the soul layer does

Beyond single-persona reactions, the soul layer supports:

- **Group chat simulation** (Sunstein 2017 polarization) — `oransim.agents.group_chat.simulate_group_chat`
- **Discourse mediators** — second-wave click/conversion impact from simulated comment threads
- **Brand memory** — multi-campaign longitudinal persona updates

Request coalescing in `oransim.agents.llm_dedup` collapses duplicate LLM calls before they hit the provider — important when running 10k agents.